In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

In [ ]:
df_hourly_sentiment=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl")
# df_hourly_sentiment.rename(columns={"Date":"Time"})


In [1]:
df_hourly_sentiment

NameError: name 'df_hourly_sentiment' is not defined

In [ ]:
df_hourly_sentiment["Date"]=df_hourly_sentiment["Date"].date()
vix=pd.read_csv("/Users/Farid Srouji/Documents/Research/master/globaldata/vix.csv")
vix.index=vix.Date
df = pd.merge(df_hourly_sentiment, vix, left_on="Date", right_index=True, how="left", sort=False)

In [ ]:
df_hourly_sentiment.Date

In [ ]:

# df_hourly_sentiment.index = df_hourly_sentiment.Date
df1_part3 = df_hourly_sentiment
idx = pd.date_range('2020-06-01', '2021-08-14')
df1_part3.index = pd.DatetimeIndex(df1_part3.index)
df_hourly_sentiment = df_hourly_sentiment.loc['2020-06-01':'2021-08-14']

In [ ]:
df_hourly_sentiment

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()] 

# Prepare data
df_hourly_sentiment['year'] = [d.year for d in df_hourly_sentiment.index]
df_hourly_sentiment['month'] = [d.strftime('%b') for d in df_hourly_sentiment.index]
df_hourly_sentiment['day'] = [d for d in df_hourly_sentiment.index.day_name()]
years = df_hourly_sentiment['year'].unique()
df_hourly_sentiment['month-year'] = [d.strftime('%b') + str(d.year) for d in df_hourly_sentiment.index]

# ANALYSIS

In [ ]:
df_hourly_sentiment

In [ ]:
df_hourly_sentiment

In [ ]:
plt.figure(figsize=(20, 7))
sns.set_theme(style="darkgrid")
# sns.colors("Blues")
ax = sns.countplot(x="Hour", data=df_hourly_sentiment,palette=['#3776ab'])
ax.set_title('Press Release Count', fontsize=18)

In [ ]:
plt.figure(figsize=(20, 7))
sns.set_theme(style="darkgrid")
ax = sns.countplot(x="Hour", hue="year", data=df_hourly_sentiment)
ax.set_title('Press Release Count', fontsize=18)

In [ ]:
#df_hourly_sentiment.Hour.value_counts().sort_values().plot(kind = 'barh',figsize=(7, 20))


In [ ]:
# Draw Plot
fig, axes = plt.subplots(1, 1, figsize=(20,7), dpi= 100)
df_hourly_sentiment.fillna(0, inplace=True)

sns.violinplot(x='Hour', y='sentiment', data=df_hourly_sentiment, ax=axes, jitter=True, zorder=1)

# Set Title
axes.set_title('Sentiment Hourly', fontsize=18)
plt.show()

In [ ]:
plt.figure(figsize=(20, 7))
sns.lineplot(x="Hour", y="neu", data=df_hourly_sentiment)

plt.show()

In [ ]:
# Draw Plot
fig, axes = plt.subplots(1, 1, figsize=(20,7), dpi= 100)
df_hourly_sentiment.fillna(0, inplace=True)

sns.violinplot(x='Hour', y='pos', data=df_hourly_sentiment, ax=axes, jitter=True, zorder=1)# data=df.loc[~df.year.isin([1991, 2008]), :])

# Set Title
axes.set_title('Positive Sentiment Hourly', fontsize=18)
plt.show()

In [ ]:
# Draw Plot
fig, axes = plt.subplots(1, 1, figsize=(20,7), dpi= 100)
df_hourly_sentiment.fillna(0, inplace=True)

sns.violinplot(x='Hour', y='neg', data=df_hourly_sentiment, ax=axes, jitter=True, zorder=1)# data=df.loc[~df.year.isin([1991, 2008]), :])

# Set Title
axes.set_title('Negative Sentiment Hourly', fontsize=18)
plt.show()

In [ ]:
# Draw Plot
fig, axes = plt.subplots(1, 1, figsize=(20,7), dpi= 100)
df_hourly_sentiment.fillna(0, inplace=True)

sns.violinplot(x='Hour', y='neu', data=df_hourly_sentiment, ax=axes, jitter=True, zorder=1)# data=df.loc[~df.year.isin([1991, 2008]), :])

# Set Title
axes.set_title('Neutral Sentiment Hourly', fontsize=18)
plt.show()

In [ ]:
df_hourly_sentiment.pivot_table(index=df_hourly_sentiment['Hour'], 
                     columns='day', 
                     values='sentiment',
                     aggfunc='sum').plot(figsize=(20,7),
                     title='Day of Week Hourly Sentiment')

In [ ]:
# Draw Plot
fig, axes = plt.subplots(1, 1, figsize=(20,7), dpi= 80)
ax_sentiment = sns.boxplot(x='Hour', y='sentiment', data=df_hourly_sentiment, ax=axes)
# ax_sentiment = sns.swarmplot(x='Hour', y='sentiment', data=df_hourly_sentiment, color=".25")

# Set Title
axes.set_title('Sentiment Hourly', fontsize=18)
plt.show()

In [ ]:
df_hourly_sentiment[['Hour','sentiment']].plot(x='Hour',
                                     y='sentiment',
                                     kind='scatter',
                                     figsize=(20,7),
                                     title='Hourly Sentiment')

In [ ]:
df = df_hourly_sentiment[["Hour", "sentiment"]]

In [ ]:

def unique(list1):
    x = np.array(list1)
    print(np.unique(x))
unique(df.Hour.values)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plots = len(df.columns)
fig, axs = plt.subplots(int(plots/2), 2, figsize=(15, 2*plots))

for Hour, ax in (unique(df.Hour.values), axs.flatten()):
        plot_acf(df.loc[Hour], ax=ax, lags=60)
        ax.set_title('Autocorrelation hour ' + str(Hour))
plt.plot()

In [ ]:
df_hourly_sentiment.Company.value_counts().sort_values().plot(kind = 'barh',figsize=(7, 20))


In [ ]:
# Columns to plot 
plot_columns = ["Open", "sentiment"]
fig, axes = plt.subplots(3,2,figsize=(25,30))
for i,col in enumerate(plot_columns):
    print("i="+str(i))
    print("col="+str(col))

    sns.regplot(col,'pos',data=df_hourly_sentiment, ax=axes[i,0]) # regression plot
    sns.kdeplot(df_hourly_sentiment[col],df_hourly_sentiment['pos'],shade=True, ax=axes[i,1]) # kernel density estimate plot